## This notebook answers whether the given article is relevant?

### Using TF-IDF Vectorizer

Here, the heuristic is that if the low frequency words varies a lot, probably the extracted paragraphs within the given article is not relevent completely...

In [1]:
import json
import os

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt 

def Tfidf(text):
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform([text])
    feature_names = vectorizer.get_feature_names_out()
    tfidf_scores = X.toarray()[0]
    return feature_names, tfidf_scores

In [3]:
def count_tfidf_scores_below_thresholds(feature_names, tfidf_scores):
    import pandas as pd
    df = pd.DataFrame({'feature': feature_names, 'tfidf_score': tfidf_scores})
    df.sort_values(by='tfidf_score', ascending=True, inplace=True)

    # count how many works have a tfidf score less than 0.1
    thr_list = [i * 0.01 for i in range(0, 11)]
    for thr in thr_list:
        print(f'Threshold: {thr}, Count: {df[df["tfidf_score"] < thr].shape[0]}')
    

In [4]:
from lxml import etree
dir_path = os.getcwd()

for file in os.listdir(dir_path):
    if file.endswith('.json'):
        with open(os.path.join(dir_path, file)) as f:
            print(f'Processing file: {file}')
            data = json.load(f)
            # load lxml data
            tree = etree.fromstring(data['ocr'].encode('utf-8'))
            text = '\n'.join(tree.xpath('//text()'))

            feature_names, tfidf_scores = Tfidf(text)
            count_tfidf_scores_below_thresholds(feature_names, tfidf_scores)

Processing file: MMHCO01_000065498_mpeg21_a0001.json
Threshold: 0.0, Count: 0
Threshold: 0.01, Count: 1047
Threshold: 0.02, Count: 1128
Threshold: 0.03, Count: 1144
Threshold: 0.04, Count: 1148
Threshold: 0.05, Count: 1149
Threshold: 0.06, Count: 1155
Threshold: 0.07, Count: 1158
Threshold: 0.08, Count: 1160
Threshold: 0.09, Count: 1160
Threshold: 0.1, Count: 1163
Processing file: MMKB32_164602050_mpeg21_a00002.json
Threshold: 0.0, Count: 0
Threshold: 0.01, Count: 0
Threshold: 0.02, Count: 197
Threshold: 0.03, Count: 197
Threshold: 0.04, Count: 226
Threshold: 0.05, Count: 242
Threshold: 0.06, Count: 242
Threshold: 0.07, Count: 247
Threshold: 0.08, Count: 247
Threshold: 0.09, Count: 251
Threshold: 0.1, Count: 252
Processing file: ddd_110532595_mpeg21_a0026.json
Threshold: 0.0, Count: 0
Threshold: 0.01, Count: 0
Threshold: 0.02, Count: 207
Threshold: 0.03, Count: 207
Threshold: 0.04, Count: 236
Threshold: 0.05, Count: 236
Threshold: 0.06, Count: 250
Threshold: 0.07, Count: 250
Threshold:

Plot TF-IDF

In [ ]:
text = "Your text here"
feature_names, tfidf_scores = Tfidf(text)

plt.bar(feature_names, tfidf_scores)
plt.xticks(rotation=90)
plt.show()

### Using LLM

In [5]:
import os
from dotenv import load_dotenv

# Load environment variables from environment.env file
load_dotenv('../../environment.env')

from groq import Groq

client = Groq(
    api_key=os.getenv('GROQ_API_KEY'),
)

def text_analysis(text):
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": "given the following text, extract the main topic in English and let me know if the complete text is coherent: " + text,
            }
        ],
        model="llama-3.3-70b-versatile",
    )

    print(chat_completion.choices[0].message.content)

In [6]:
import json, os
from lxml import etree
dir_path = os.getcwd()

for file in os.listdir(dir_path):
    if file.endswith('.json'):
        with open(os.path.join(dir_path, file)) as f:
            print(f'Processing file: {file}')
            data = json.load(f)
            # load lxml data
            tree = etree.fromstring(data['ocr'].encode('utf-8'))
            text = '\n'.join(tree.xpath('//text()'))

            text_analysis(text)
        print('-----------------------------')

Processing file: MMHCO01_000065498_mpeg21_a0001.json
The main topic of the text in English is: News from the Netherlands, including politics, government decisions, and various events.

As for the coherence of the text, it appears to be a collection of news articles or reports from various sources, and as such, it is not a single, cohesive text. The text jumps between different topics, such as government decisions, appointments, and events, without a clear narrative thread or logical connection between them. However, each individual section or article within the text appears to be coherent and well-structured, suggesting that the text is a compilation of separate news items.
-----------------------------
Processing file: MMKB32_164602050_mpeg21_a00002.json
The main topic of the given text in English is: **Official Decrees and Appointments in the Dutch East Indies (Netherlands-Indies)**.

As for the coherence of the text, it appears to be a collection of official documents, including dec

Is the text relevant to the historical event?

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from environment.env file
load_dotenv('../../environment.env')

from groq import Groq

client = Groq(
    api_key=os.getenv('GROQ_API_KEY'),
)

def text_analysis(text, event):
    prompt = f"""
        Given the historical event: {event}, determine whether given text describe any involvement with the event.
        
        Return your answer in this format:

        Answer: Yes/No

        Explanation: one or two sentences explaining why

        The text is: {text} 
        """
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        model="llama-3.3-70b-versatile",
    )
    print(chat_completion.choices[0].message.content)

GroqError: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable

In [8]:
import json, os
from lxml import etree
dir_path = os.getcwd()

for file in os.listdir(dir_path):
    if file.endswith('.json'):
        with open(os.path.join(dir_path, file)) as f:
            print(f'Processing file: {file}')
            data = json.load(f)
            # load lxml data
            tree = etree.fromstring(data['ocr'].encode('utf-8'))
            text = '\n'.join(tree.xpath('//text()'))
            event = data['event_title']
            text_analysis(text, event)
        print('-----------------------------')

Processing file: MMHCO01_000065498_mpeg21_a0001.json
Answer: Yes

Explanation: The text mentions "ter zake der 2e expeditie naar Nias in 1863" which indicates that the event is being referenced in the context of awarding a title to someone, specifically the officier van gezondheid der 1e klasse bij het leger in Nederlandsch Indië E. E. M. Helmkampf, who is being made a ridder der 4e klasse van de Militaire Willems-orde.
-----------------------------
Processing file: MMKB32_164602050_mpeg21_a00002.json
Answer: Yes

Explanation: The text mentions the "2e expeditie naar Nias in 1863" as the reason for the toekenning van eervolle belooningen (award of honors) and the benoeming (appointment) of kapitein der infanterie Q. V. Hoek as Ridder der 4e klasse van de militaire Willems-orde (Knight of the 4th class of the Military Order of William). This indicates that the text describes an involvement with the historical event of the 2nd expedition to Nias.
-----------------------------
Processing 

### Using ollama

In [11]:
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(model='llama3', messages=[
  {
    'role': 'user',
    'content': 'Why is the sky blue?',
  },
])
print(response['message']['content'])
# or access fields directly from the response object
print(response.message.content)

A classic question!

The sky appears blue because of a phenomenon called Rayleigh scattering, named after the British physicist Lord Rayleigh. He discovered that when sunlight enters Earth's atmosphere, it encounters tiny molecules of gases like nitrogen (N2) and oxygen (O2). These molecules scatter the light in all directions, but they scatter shorter wavelengths (like blue and violet) more than longer wavelengths (like red and orange).

Here's what happens:

1. Sunlight consists of a broad spectrum of colors, including all the colors of the rainbow.
2. When this sunlight enters the Earth's atmosphere, it encounters tiny molecules of gases like N2 and O2.
3. These molecules scatter the light in all directions, but they scatter shorter wavelengths (like blue and violet) more than longer wavelengths (like red and orange).
4. This scattering effect is stronger for shorter wavelengths, which means that the blue and violet parts of the sunlight are scattered more than the other colors.
5. 

### Sequence Similarity

In [ ]:
!pip install sentence_transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util

def is_related_to_event(text, event_description, threshold=0.5):
    model = SentenceTransformer("all-MiniLM-L6-v2")
    text_emb = model.encode(text, convert_to_tensor=True)
    event_emb = model.encode(event_description, convert_to_tensor=True)
    similarity = util.cos_sim(text_emb, event_emb).item()
    return similarity >= threshold, similarity



/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/sarah_shoilee/anaconda3/envs/delpher/lib/python3.9/runpy.py", line 87, in _run_

In [ ]:
import json, os
from lxml import etree
dir_path = 'sample_data/DST'

for file in os.listdir(dir_path):
    if file.endswith('.json'):
        with open(os.path.join(dir_path, file)) as f:
            print(f'Processing file: {file}')
            data = json.load(f)
            # load lxml data
            tree = etree.fromstring(data['ocr'].encode('utf-8'))
            text = '\n'.join(tree.xpath('//text()'))
            event = data['event_title']
            related, score = is_related_to_event(text, event)
            print(related, score)
        print('-----------------------------')